In [1]:
import rclpy
from custom_interfaces.srv import GetMushroom

from rclpy.node import Node
import numpy as np
from sensor_msgs.msg import JointState
from geometry_msgs.msg import PoseStamped
from std_msgs.msg import String as RosString
import random
import time
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32 as ROSFloat
from custom_interfaces.srv import GripperAngleService

from custom_interfaces.srv import RrtRequest
from custom_interfaces.msg import ConfigRep
from custom_interfaces.srv import RobotListAngleMove
from sensor_msgs.msg import PointCloud2
from sensor_msgs_py import point_cloud2 

from scipy.spatial.transform import Rotation as R
from rrt import RRT, forward_kinematics
from alignment_ultis import get_transform,transform_mushroom_to_global,get_tip_positions,Mushroom,tip_data

In [2]:
import pickle

# Load the precomputed forward-kinematics cache.
# Only load pickle files from sources you trust.
with open("fk.cahce", "rb") as file:
    fk_cache = pickle.load(file)

lengths = np.asarray(fk_cache["lengths"], dtype=float)
limits = np.asarray(fk_cache["limits"], dtype=float)
angles = np.asarray(fk_cache["angles"], dtype=float)
fks = np.asarray(fk_cache["fks"], dtype=float)


In [3]:


class ControlNode(Node):
    def __init__(self):
        super().__init__("control_node")
        self.future_side_camera=None
        self.future_top_camera=None

        self.cli_side=self.create_client(GetMushroom,"/get_side_mushroom_location")
        self.cli_top=self.create_client(GetMushroom,"/get_mushroom_location")
        
        self.close_gripper=self.create_client(GripperAngleService,"/close_gripper_angle")
        self.griper_task_publisher = self.create_publisher(RosString,"/gripper_task",10)
        
        self.joint_publisher=self.create_publisher(JointState,"/joint_states_change_request",10)
        self.create_subscription(JointState,'/joint_states',self.update,10)
        
        self.joints={}
        self.req = GetMushroom.Request()
        
        
        self.top_mushroom_position=[]
        self.pub_=self.create_publisher(PoseStamped,"goal_pose",10)
        self.create_subscription(RosString,"/path_plan_status",self.sync_ros,10)
        self.is_robot_free = ""
        self.started="complete"
        self.status=None
        self.change_z_publisher_ = self.create_publisher(ROSFloat,"/z_change_request",10)
    
        self.client = self.create_client(RrtRequest,'/rrt_service/solver/rrt_solver')
        self.move_robot_client = self.create_client(RobotListAngleMove,"/robot_service/move/config_rep_list")
        
        self.create_subscription(PointCloud2,"/obstcale_points",self.update_obs_points,10)
        # self.create_subscription(JointState,"/joint_states",self.update_joint_poistion,10)
        self.laser_update=False
        self.jp_update=False
        self.joints={}
        self.old_obstacles=[]

    def update_obs_points(self,msg):
        points = point_cloud2.read_points(msg, field_names=["x", "y", "z"], skip_nans=True)

        points=np.asarray([[x,y] for x,y,z in points])
        self.points=points.copy()
        if len(self.joints)<1:
            return
        joint_angles = np.array([self.joints['Angle1'], self.joints['Angle2'], self.joints['Angle3']], dtype=float)
        cumulative = np.cumsum(joint_angles)
        segment_vectors = np.column_stack((lengths * np.cos(cumulative), lengths * np.sin(cumulative)))
        pos = np.vstack((np.zeros(2), np.cumsum(segment_vectors, axis=0)))

        off_point=self.points-pos[-2]
        off_robot=pos-pos[-2]
        norms = np.linalg.norm(off_point, axis=1, keepdims=True)
        end_points1 = off_point*1.25 / norms
        end_points2 = off_point*1.1
        self.laser_update=True
        self.end_points1=[]
        self.end_points2=[]
        for i,j,k in zip(off_point,end_points1,end_points2):
            
            y=[j,k][np.argmin(np.linalg.norm([j,k],axis=1))]
            # obstacles.append(LineString([i+pos[-2],y+pos[-2]]).buffer(0.02))
            x=i+pos[-2]
            y=y+pos[-2]
            self.end_points1.append(x)
            self.end_points2.append(y)

    def send_request(self, obstacles, target, angles):

        request = RrtRequest.Request()
        request.obstacles = np.asarray(
            obstacles,
            dtype=float
        ).flatten().tolist()

        request.target = np.asarray(
            target,
            dtype=float
        ).flatten().tolist()

        request.angles = np.asarray(
            angles,
            dtype=float
        ).flatten().tolist()

        # self.get_logger().info(
        #     f'Sending target: {request.target}'
        # )

        return self.client.call_async(request)
    
    
    def send_request_to_move_robot(self, list_of_angles):

        request = RobotListAngleMove.Request()

        request.list_of_angles = list_of_angles

        return self.move_robot_client.call_async(request)
    
    
    
    def send_gripper_request(self,a1,a2,a3):
        request=GripperAngleService.Request()
        request.angle1=float(a1)
        request.angle2=float(a2)
        request.angle3=float(a3)
        return self.close_gripper.call_async(request)
    
    def gripper_open(self):
        msg=RosString()
        msg.data="open"
        self.griper_task_publisher.publish(msg)
        # return self.open_cli.call_async(Trigger.Request())
    def gripper_close(self):
        msg=RosString()
        msg.data="close"
        self.griper_task_publisher.publish(msg)

    def sync_ros(self,msg):
        self.is_robot_free=msg.data
        if self.status != "completed":
            if self.is_robot_free == "running":
                self.status="running"
                self.started=True
            
            if self.started and self.is_robot_free == "sucess":
                self.status="completed"
    
    def publish_goal_target(self,target):
        self.status=""
        x,y=target
        msg=PoseStamped()
        msg.pose.position.x=float(x)
        msg.pose.position.y=float(y)
        self.pub_.publish(msg)
        # time.sleep(1)
        self.started=False
        
        # return self.is_robot_free
        
    def update(self,msg):
        self.updated=True
        self.joints={i:j for i,j in zip(msg.name,msg.position)}
        
        
    def move_z(self,z):
        # msg=JointState()
        # msg.name=list(self.joints.keys())
        # msg.position=list(self.joints.values())
        # msg.position[0]=z
        # self.joint_publisher.publish(msg)
        msg=ROSFloat()
        msg.data=float(z)
        self.change_z_publisher_.publish(msg)
        

    
    def get_mushroom_side_camera(self):
       
        if not self.cli_side.wait_for_service(timeout_sec=5.0):
            self.get_logger().error("Service not available")
            return None

        self.future_side_camera = self.cli_side.call_async(self.req)
        return self.future_side_camera
    
    def get_mushroom_top_camera(self):
       
        if not self.cli_top.wait_for_service(timeout_sec=5.0):
            self.get_logger().error("Service not available")
            return None

        self.future_top_camera = self.cli_top.call_async(self.req)
        return self.future_top_camera

    def goto_z(self, z, sleep_time=0.1):
        self.laser_update = False
        while not self.laser_update:
            rclpy.spin_once(self)

        obstacles = np.array([
            [i, j] for i, j in zip(self.end_points1, self.end_points2)
        ])


        # target = [0.5, -0.3]

        angles = np.array([self.joints["Angle1"],self.joints["Angle2"],self.joints["Angle3"],], dtype=float)
        list_of_angles=[]
        rep = ConfigRep()
        rep.angle1 = float(angles[0])
        rep.angle2 = float(angles[1])
        rep.angle3 = float(angles[2])
        rep.z = float(z*1000)
        rep.speed = float(300)
        list_of_angles.append(rep)

        if list_of_angles:
            future=self.send_request_to_move_robot(list_of_angles)
            rclpy.spin_until_future_complete(self, future)
        time.sleep(sleep_time)
        
    def goto_target(self, target, sleep_time=0.1, speed=None):

        obstacle_points = self.update_obstcales()
        # ----------------------------------
        # 2. Get current joint configuration
        # ----------------------------------
        angles = np.array([
            self.joints["Angle1"],
            self.joints["Angle2"],
            self.joints["Angle3"]
        ], dtype=float)

        # ----------------------------------
        # 3. Initialize RRT planner
        # ----------------------------------
        time_start = time.perf_counter()

        rrt = RRT(
            start=angles,
            target=target,
            lengths=lengths,
            limits=limits,
            obstacles=obstacle_points,
            obstacle_buffer=0.03,
            edge_resolution=0.03,
            step_size=0.15,
            goal_bias=0.2,
            goal_tolerance=0.05,
            ik_tolerance=0.001,
            max_iterations=10000,
            k=100,
            min_ee_radius=0.1,
        )

        # ----------------------------------
        # 4. Plan and smooth path
        # ----------------------------------
        path = rrt.build_tree()

        if path is None or len(path) == 0:
            print("RRT planning failed: no path found")
            return False

        s_path = rrt.smooth_path(path)

        if s_path is not None and len(s_path) > 0:
            path = s_path

        path = np.asarray(path, dtype=float).reshape(-1, 3)

        # print(
        #     "path planning time:",
        #     time.perf_counter() - time_start
        # )

        # print("Number of configurations:", len(path))

        # ----------------------------------
        # 5. Prepare movement configurations
        # ----------------------------------
        time_start = time.perf_counter()

        if speed is None:
            speed = 200

        list_of_angles = []
        prev = angles.copy()

        for val in path:
            if np.allclose(val, prev, atol=1e-8, rtol=0):
                continue

            rep = ConfigRep()
            rep.angle1 = float(val[0])
            rep.angle2 = float(val[1])
            rep.angle3 = float(val[2])
            rep.z = 0.0
            rep.speed = float(speed)

            list_of_angles.append(rep)
            prev = val

        if not list_of_angles:
            print("Robot is already at the target configuration")
            return False

        # ----------------------------------
        # 6. Execute robot movement
        # ----------------------------------
        future = self.send_request_to_move_robot(list_of_angles)

        rclpy.spin_until_future_complete(self, future)

        if future.result() is None:
            raise RuntimeError("Robot movement service returned no result")

        time.sleep(sleep_time)

        # print(
        #     "movement time:",
        #     time.perf_counter() - time_start
        # )

        return True
    
    def update_obstcales(self):
        self.laser_update=False
        while not self.laser_update:
            rclpy.spin_once(self)
        obstacles = np.array([[i, j] for i, j in zip(self.end_points1, self.end_points2)])
        angles = [self.joints[key] for key in ("Angle1", "Angle2", "Angle3")]

        self.old_obstacles = np.asarray(self.old_obstacles).reshape(-1, 2, 2)
        fk = forward_kinematics(angles, lengths)
        laser_pos = fk[-2]
        heading = np.sum(angles)

        # 2. Midpoints of previous obstacles
        points = self.old_obstacles.mean(axis=1)

        # 3. Vectors from laser to previous obstacles
        vectors = points - laser_pos

        # 4. Angles relative to laser
        theta = np.arctan2(vectors[:, 1], vectors[:, 0])

        # 5. Angular difference from current heading
        diff = np.arctan2(
            np.sin(theta - heading),
            np.cos(theta - heading)
        )

        # 6. Select previous obstacles within ±22 degrees
        mask = np.abs(diff) <= np.deg2rad(22)

        selected = self.old_obstacles[mask]
        complete_obstacles=[]
        complete_obstacles.extend(obstacles.tolist())
        complete_obstacles.extend(selected.tolist())
        self.old_obstacles=complete_obstacles.copy()

        return complete_obstacles


In [4]:
rclpy.init()

In [5]:
node = ControlNode()

Movement helper moved into `ControlNode` above.


In [ ]:
node.send_gripper_request(0,0,0)

In [ ]:
node.gripper_close()

In [6]:
from rrt import RRT, forward_kinematics


Movement helper moved into `ControlNode` above.


In [7]:
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image
from cv_bridge import CvBridge
import cv2
from datetime import datetime


class CameraCapture(Node):
    def __init__(self):
        super().__init__("camera_capture")

        self.bridge = CvBridge()
        self.image_received = False

        self.subscription = self.create_subscription(
            Image,
            "/color_image",
            self.image_callback,
            10
        )

    def image_callback(self, msg):
        image = self.bridge.imgmsg_to_cv2(msg, "bgr8")

        filename ="drop_off_images/" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".jpg"
        cv2.imwrite(filename, image)

        self.get_logger().info(f"Saved: {filename}")
        self.image_received = True

    def run(self):
        count = 0

        while rclpy.ok() and not self.image_received and count < 100:
            rclpy.spin_once(self, timeout_sec=0.1)
            count += 1

        if not self.image_received:
            self.get_logger().warning("No image received")

        self.destroy_node()





In [8]:

import os
import time
import random
import uuid
import signal
import subprocess
from pathlib import Path

import numpy as np
import cv2
import rclpy


# ============================================================
# 1. ROS2 BAG RECORDING
# ============================================================

def start_bag_recording(run_number):
    run_uuid = uuid.uuid4().hex[:8]

    bag_dir = Path("bag_data")
    bag_dir.mkdir(parents=True, exist_ok=True)

    bag_path = bag_dir / (
        f"lab_test4_run_{run_number:03d}_{run_uuid}"
    )

    process = subprocess.Popen(
        [
            "ros2",
            "bag",
            "record",
            "-a",
            "-o",
            str(bag_path)
        ],
        start_new_session=True
    )

    # Allow topic discovery before harvesting
    time.sleep(1.0)

    if process.poll() is not None:
        raise RuntimeError(
            f"ROS2 bag recorder exited early: {process.returncode}"
        )

    print(f"Recording ROS2 bag: {bag_path}")
    return process, bag_path


def stop_bag_recording(process, bag_path):
    if process is None:
        return

    if process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)

        try:
            process.wait(timeout=10)

        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)

            try:
                process.wait(timeout=5)

            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()

    print(f"ROS2 bag stopped: {bag_path}")


# ============================================================
# 2. MUSHROOM DETECTION
# ============================================================

def detect_mushrooms(camera="side"):
    if camera == "side":
        future = node.get_mushroom_side_camera()
        rclpy.spin_until_future_complete(node, future)

        return [
            mush
            for msg in future.result().mushrooms
            if len(msg.points_flat) > 0
            if (mush := Mushroom(msg)).distance <= 0.8
        ]

    future = node.get_mushroom_top_camera()
    T = get_transform()
    rclpy.spin_until_future_complete(node, future)

    return [
        mush
        for msg in future.result().mushrooms
        if np.isfinite(
            (mush := Mushroom(msg, T, msg.mask)).x
        )
        and np.isfinite(mush.distance_from_gripper)
    ]


# ============================================================
# 3. CALIBRATION
# ============================================================

def calibrate_mushroom(
    picked_mushroom,
    max_iterations=3,
    tolerance=0.015
):
    for cal in range(max_iterations):
        start = time.perf_counter()

        angle = sum(
            node.joints[key]
            for key in ("Angle1", "Angle2", "Angle3")
        )

        offset = -0.0165 * np.array([
            np.cos(angle),
            np.sin(angle)
        ])

        target = np.array([
            picked_mushroom.x,
            picked_mushroom.y
        ]) + offset

        node.goto_target(
            target.tolist(),
            sleep_time=0.25,
            speed=170
        )

        mushrooms = detect_mushrooms("top")

        if not mushrooms:
            print("No mushroom detected during calibration")
            return None

        selected = min(
            mushrooms,
            key=lambda mush: picked_mushroom - mush
        )

        distance = picked_mushroom - selected
        picked_mushroom = selected

        print(
            f"step4_{cal} calibration: "
            f"{time.perf_counter() - start:.3f}s"
        )

        if distance < tolerance:
            print("Calibration complete")
            return picked_mushroom

    print("Calibration tolerance not reached")
    return picked_mushroom


# ============================================================
# 4. GRIPPER MASK
# ============================================================

def prepare_gripper_mask(mushroom):
    mask = mushroom.mask
    depth = np.max(mushroom.points_original[:, 2])

    scale = depth / (depth + 0.081)

    h, w = mask.shape
    H = max(1, int(h * scale))
    W = max(1, int(w * scale))

    resized = cv2.resize(
        mask,
        (W, H),
        interpolation=cv2.INTER_NEAREST
    )

    result = np.zeros_like(mask)

    src_y = max(0, (H - h) // 2)
    src_x = max(0, (W - w) // 2)

    dst_y = max(0, (h - H) // 2)
    dst_x = max(0, (w - W) // 2)

    copy_h = min(h, H)
    copy_w = min(w, W)

    result[
        dst_y:dst_y + copy_h,
        dst_x:dst_x + copy_w
    ] = resized[
        src_y:src_y + copy_h,
        src_x:src_x + copy_w
    ]

    return result, depth


# ============================================================
# 5. HARVEST EXECUTION
# ============================================================

def execute_harvest(run_number):
    total_start = time.perf_counter()
    stage_start = total_start

    # Step 1: Side camera detection
    mushrooms = detect_mushrooms("side")

    if not mushrooms:
        print("No mushrooms detected")
        return False

    selected = random.choice(mushrooms)

    print(
        f"step1 side inference time: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 2: Initial movement
    node.goto_target(
        [selected.x + 0.05, selected.y],
        sleep_time=0.3,
        speed=500
    )

    print(
        f"step2 move time: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 3: Top camera detection
    mushrooms = detect_mushrooms("top")

    if not mushrooms:
        print("No mushrooms detected from top camera")
        return False

    picked_mushroom = min(
        mushrooms,
        key=lambda mush: mush.distance_from_gripper
    )

    print(
        f"step3 top inference: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 4: Calibration
    picked_mushroom = calibrate_mushroom(
        picked_mushroom
    )

    if picked_mushroom is None:
        return False

    print(
        f"step4 calibration: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 5: Gripper alignment and pickup
    mask, depth = prepare_gripper_mask(
        picked_mushroom
    )

    positions = get_tip_positions(
        mask,
        tip_data,
        tolerance=30
    )

    node.send_gripper_request(*positions)
    time.sleep(0.1)

    node.goto_z(depth + 0.035)
    node.gripper_close()
    time.sleep(2.0)

    print(
        f"step5 pickup time: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 6: Drop-off
    node.goto_z(0)
    node.goto_target(
        [0.275, -0.25],
        speed=500
    )

    print(
        f"step6 drop off: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    print(
        f"run time: "
        f"{time.perf_counter() - total_start:.3f}s"
    )

    return True


# ============================================================
# 6. RUN HARVEST WITH OPTIONAL BAG RECORDING
# ============================================================

def harvest_once(run_number, log=False):
    bag_process = None
    bag_path = None

    try:
        if log:
            bag_process, bag_path = start_bag_recording(
                run_number
            )

        success = execute_harvest(run_number)

        if success:
            node_camera = CameraCapture()

            try:
                node_camera.run()
            finally:
                node_camera.destroy_node()

        return success

    finally:
        # Open gripper while recorder is still active
        try:
            node.gripper_open()
        finally:
            if bag_process is not None:
                stop_bag_recording(
                    bag_process,
                    bag_path
                )





In [9]:
import numpy as np


def twist(angles, delta_angle, lengths,tolerance=1e-5, max_iterations=40):

    angles = np.asarray(angles, dtype=float)
    l1, l2, l3 = lengths
    t1, t2, t3 = angles

    if abs(delta_angle) < tolerance:
        return angles.copy()

    # Current end-effector position
    phi0 = np.sum(angles)

    x = (
        l1 * np.cos(t1)
        + l2 * np.cos(t1 + t2)
        + l3 * np.cos(phi0)
    )

    y = (
        l1 * np.sin(t1)
        + l2 * np.sin(t1 + t2)
        + l3 * np.sin(phi0)
    )

    def solve(delta):

        phi = phi0 + delta

        wx = x - l3 * np.cos(phi)
        wy = y - l3 * np.sin(phi)

        c2 = (wx**2 + wy**2 - l1**2 - l2**2) / (2 * l1 * l2)

        if abs(c2) > 1 + 1e-10:
            return None

        c2 = np.clip(c2, -1, 1)

        solutions = []

        for sign in [1, -1]:

            theta2 = sign * np.arccos(c2)

            theta1 = (
                np.arctan2(wy, wx)
                - np.arctan2(
                    l2 * np.sin(theta2),
                    l1 + l2 * np.cos(theta2)
                )
            )

            theta3 = phi - theta1 - theta2

            solution = np.array([theta1, theta2, theta3])

            # Closest equivalent joint angles
            solution = angles + (
                (solution - angles + np.pi) % (2 * np.pi) - np.pi
            )

            solutions.append(solution)

        return min(
            solutions,
            key=lambda s: np.linalg.norm(s - angles)
        )

    # Try requested twist directly
    result = solve(delta_angle)

    if result is not None:
        return result

    # Find largest achievable twist in the requested direction
    low, high = 0.0, 1.0
    best = angles.copy()

    for _ in range(max_iterations):

        mid = (low + high) / 2

        result = solve(delta_angle * mid)

        if result is not None:
            best = result
            low = mid
        else:
            high = mid

        if (high - low) * abs(delta_angle) < tolerance:
            break

    return best

In [10]:
total_start = time.perf_counter()
stage_start = total_start

# Step 1: Side camera detection
mushrooms = detect_mushrooms("side")
if len(mushrooms)>0:
    selected = random.choice(mushrooms)
    print(
        f"step1 side inference time: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 2: Initial movement
    node.goto_target(
        [selected.x + 0.05, selected.y],
        sleep_time=0.1,
        speed=500
    )
else:
    print("no mushroom side cam")
    node.goto_target(
        [0.7, 0],
        sleep_time=0.1,
        speed=500
    )


print(
    f"step2 move time: "
    f"{time.perf_counter() - stage_start:.3f}s"
)
stage_start = time.perf_counter()

# Step 3: Top camera detection
mushrooms = detect_mushrooms("top")

picked_mushroom = min(
    mushrooms,
    key=lambda mush: mush.distance_from_gripper
)

print(
    f"step3 top inference: "
    f"{time.perf_counter() - stage_start:.3f}s"
)
stage_start = time.perf_counter()

# Step 4: Calibration
picked_mushroom = calibrate_mushroom(
    picked_mushroom
)

print(
    f"step4 calibration: "
    f"{time.perf_counter() - stage_start:.3f}s"
)
stage_start = time.perf_counter()

# Step 5: Gripper alignment and pickup
mask, depth = prepare_gripper_mask(
    picked_mushroom
)

positions = get_tip_positions(
    mask,
    tip_data,
    tolerance=20
)
node.send_gripper_request(*positions)
node.goto_z(depth + 0.035)

# time.sleep(0.1)


node.gripper_close()
node.updated=False
while not node.updated:
    rclpy.spin_once(node)
time.sleep(0.75)
angles = [node.joints[key] for key in ("Angle1", "Angle2", "Angle3")]
twisted_angles=twist(angles,0.3,lengths)
list_of_angles=[]
rep = ConfigRep()
rep.angle1 = float(twisted_angles[0])
rep.angle2 = float(twisted_angles[1])
rep.angle3 = float(twisted_angles[2])
rep.z = float((depth + 0.035)*1000)
rep.speed = float(80)
list_of_angles.append(rep)

if list_of_angles:
    future=node.send_request_to_move_robot(list_of_angles)
    rclpy.spin_until_future_complete(node, future)
node.goto_z(0)
# time.sleep(0.5)
node.goto_target([0.075,-0.35],sleep_time=0.01,speed=500)
node.gripper_open()
time.perf_counter()-total_start

[ERROR] [1794278073.325900218] [control_node]: Service not available


AttributeError: 'NoneType' object has no attribute 'add_done_callback'

In [85]:
open=False

In [90]:
if open:
    node.gripper_open()
    open=False
else:
    node.gripper_close()
    open=True

In [91]:
node.goto_z(0)
node.gripper_open()

In [82]:
list_of_angles=[]
rep = ConfigRep()
rep.angle1 = float(angles[0])
rep.angle2 = float(angles[1])
rep.angle3 = float(angles[2])
rep.z = float((0)*1000)
rep.speed = float(100)
list_of_angles.append(rep)

if list_of_angles:
    future=node.send_request_to_move_robot(list_of_angles)
    rclpy.spin_until_future_complete(node, future)


In [ ]:
twisted_angles=twist(angles,0.15,lengths)
list_of_angles=[]
rep = ConfigRep()
rep.angle1 = float(twisted_angles[0])
rep.angle2 = float(twisted_angles[1])
rep.angle3 = float(twisted_angles[2])
rep.z = float((0)*1000)
rep.speed = float(100)
list_of_angles.append(rep)

if list_of_angles:
    future=node.send_request_to_move_robot(list_of_angles)
    rclpy.spin_until_future_complete(node, future)


In [ ]:
node.gripper_open()

In [ ]:
node.goto_z(0)

In [ ]:
list_of_angles=[]
rep = ConfigRep()
rep.angle1 = float(twisted_angles[0])
rep.angle2 = float(twisted_angles[1])
rep.angle3 = float(twisted_angles[2])
rep.z = float((depth + 0.035)*1000)
rep.speed = float(100)
list_of_angles.append(rep)

if list_of_angles:
    future=node.send_request_to_move_robot(list_of_angles)
    rclpy.spin_until_future_complete(node, future)

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.plot(*forward_kinematics(angles,lengths).T)

plt.plot(*forward_kinematics(twisted_angles,lengths).T)

In [ ]:
node.goto_target([0.175,-0.3])
node.gripper_open()

In [ ]:
harvest_once(1,False)

In [ ]:
# ============================================================
# 7. MAIN LOOP
# ============================================================

for run in range(20):
    print(f"\nRun {run + 1}/20")

    success = harvest_once(
        run_number=run,
        log=True
    )

    print(f"Harvest success: {success}")
    print("_" * 50)

In [ ]:
node.gripper_open()

In [ ]:
mushrooms = detect_mushrooms("side")
print("first selected mushroom")
selected = random.choice(mushrooms)
print(selected)

In [ ]:
mushrooms = detect_mushrooms("side")
print("first selected mushroom")
if len(mushrooms)>0:
    selected = random.choice(mushrooms)
    print(selected)
    node.goto_target(
            [selected.x, selected.y],
            sleep_time=0.3,
            speed=350
        )
else:
    print("side camera failed")
    node.goto_target(
            [0.5,0],
            sleep_time=0.3,
            speed=350
        )

In [ ]:
mushrooms = detect_mushrooms("top")
picked_mushroom = min(
        mushrooms,
        key=lambda mush: mush.distance_from_gripper
    )
print(picked_mushroom)

In [ ]:
angle = sum(
            node.joints[key]
            for key in ("Angle1", "Angle2", "Angle3")
        )
offset = -0.01 * np.array([
    np.cos(angle),
    np.sin(angle)
])

target = np.array([
    picked_mushroom.x-0.01,
    picked_mushroom.y
]) + offset

node.goto_target(
    target.tolist(),
    sleep_time=0.25,
    speed=170
)


In [ ]:
mask, depth = prepare_gripper_mask(
        picked_mushroom
    )



node.goto_z(depth + 0.035)

In [ ]:
node.send_gripper_request(1.5,1.5,1.5)

In [ ]:
node.gripper_close()

In [ ]:
node.goto_z(0)
node.goto_target([0.1750,-0.3])

In [ ]:
node.gripper_open()

In [ ]:

    print(
        f"step1 side inference time: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 2: Initial movement
    
    print(
        f"step2 move time: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()

    # Step 3: Top camera detection
    mushrooms = detect_mushrooms("top")

    if not mushrooms:
        print("No mushrooms detected from top camera")
        return False

    picked_mushroom = min(
        mushrooms,
        key=lambda mush: mush.distance_from_gripper
    )

    print(
        f"step3 top inference: "
        f"{time.perf_counter() - stage_start:.3f}s"
    )
    stage_start = time.perf_counter()